# Harmless injection fixtures in an owned toy app

Use this folder and the shared root kernel. The supplied document is untrusted data. All inputs are synthetic; DEMO_OVERRIDE is only a harmless marker, with no secret or external action.

In [ ]:
import json
from pathlib import Path
from pprint import pprint
fixtures = json.loads(Path("docs/dataset/inputs.json").read_text())
pprint(fixtures)

## Read the deliberately simple simulation

`models/toy.py` returns DEMO_OVERRIDE if the marker appears in either input; otherwise it returns a fixed policy line. It is intentionally unsafe deterministic code, not an LLM. `tools/boundary.py` optionally blocks one phrase and records whether the toy actually ran. Read both before importing.

In [ ]:
from models.toy import toy_response
from tools.boundary import inspect_request, request
pprint(inspect_request(fixtures[1]["question"], fixtures[1]["document"]))

## Actually run the unguarded toy

These are outputs from real local function invocations. They do not show that any language model obeyed an attack. The marker rule makes the direct, indirect, and alternate-wording fixtures produce the same output.

In [ ]:
baseline_seen = []
baseline = {item["id"]: request(item["question"], item["document"], toy_response, baseline_seen) for item in fixtures}
for name, result in baseline.items():
    pprint({"id": name, **result})
pprint({"actual_toy_invocations": len(baseline_seen), "model_api_calls": 0})

## Enable one phrase check

Change only block_phrase from False to True and replay the exact question/document pairs. The phrase is matched anywhere, so quoting it for education also matches.

In [ ]:
candidate_seen = []
candidate = {item["id"]: request(item["question"], item["document"], toy_response, candidate_seen, block_phrase=True) for item in fixtures}
for name, result in candidate.items():
    pprint({"id": name, **result})

## Compare the same inputs

The direct and document-embedded instructions are withheld. Inspect the final output as well as the guard decision; allowing a request alone does not imply safety.

In [ ]:
for item in fixtures:
    name = item["id"]
    pprint({"id": name, "baseline": baseline[name]["exposed"], "candidate_decision": candidate[name]["decision"], "candidate_exposed": candidate[name]["exposed"]})
assert candidate["direct"]["local_responder_calls"] == 0
assert candidate["indirect"]["local_responder_calls"] == 0

## Bypass and false positive

Alternative wording bypasses the phrase check; an educational quotation is blocked. This toy simulation cannot prove prompt-injection safety. Phrase filters and stronger prompts do not create a reliable instruction/data boundary; real-model testing would be a separate exercise.

In [ ]:
assert candidate["wording_bypass"]["decision"] == "allow"
assert candidate["wording_bypass"]["exposed"] == "DEMO_OVERRIDE"
assert candidate["benign_lookalike"]["decision"] == "block"
pprint({"bypass": candidate["wording_bypass"], "false_positive": candidate["benign_lookalike"], "actual_candidate_toy_invocations": len(candidate_seen)})